# Run the recommendation pipeline

This job executes the notebooks in dependency order:
1. `00_get_data.ipynb` collects Twitch data and exports snapshots
2. `01_set_baseline.ipynb` creates the weighted baseline ranking
3. `02_compare_models.ipynb` runs staged filtering and the model pipeline
4. `03_metadata_embeddings.ipynb` trains metadata embeddings and cosine-similarity recommendations

Set Twitch and Postgres environment variables before running. The job skips interactive Twitch OAuth; provide `TWITCH_USER_ACCESS_TOKEN` for user-specific follows and VODs.

In [4]:
import importlib
import os
from pathlib import Path

nbformat = importlib.import_module("nbformat")
NotebookClient = importlib.import_module("nbclient.client").NotebookClient

notebooks_dir = Path.cwd() if Path.cwd().name == "notebooks" else Path.cwd() / "notebooks"
pipeline = [
    notebooks_dir / "00_get_data.ipynb",
    notebooks_dir / "01_set_baseline.ipynb",
    notebooks_dir / "02_compare_models.ipynb",
    notebooks_dir / "03_metadata_embeddings.ipynb",
]
missing = [str(path) for path in pipeline if not path.exists()]
if missing:
    raise FileNotFoundError(f"Missing pipeline notebooks: {missing}")

previous_job_mode = os.environ.get("TWITCH_PIPELINE_JOB")
os.environ["TWITCH_PIPELINE_JOB"] = "1"
try:
    for notebook_path in pipeline:
        print(f"Running {notebook_path.name}...")
        with notebook_path.open("r", encoding="utf-8") as handle:
            notebook = nbformat.read(handle, as_version=4)
        client = NotebookClient(
            notebook,
            timeout=None,
            kernel_name="python3",
            resources={"metadata": {"path": str(notebooks_dir)}},
        )
        client.execute()
        with notebook_path.open("w", encoding="utf-8") as handle:
            nbformat.write(notebook, handle)
        print(f"Completed {notebook_path.name}")
finally:
    if previous_job_mode is None:
        os.environ.pop("TWITCH_PIPELINE_JOB", None)
    else:
        os.environ["TWITCH_PIPELINE_JOB"] = previous_job_mode

print("Recommendation pipeline completed successfully.")

Running 00_get_data.ipynb...
Completed 00_get_data.ipynb
Running 01_set_baseline.ipynb...
Completed 01_set_baseline.ipynb
Running 02_compare_models.ipynb...
Completed 02_compare_models.ipynb
Running 03_metadata_embeddings.ipynb...
Completed 03_metadata_embeddings.ipynb
Recommendation pipeline completed successfully.


In [5]:
from pathlib import Path

import pandas as pd
from IPython.display import display

notebooks_dir = Path.cwd() if Path.cwd().name == "notebooks" else Path.cwd() / "notebooks"
model_outputs = {
    "Baseline model": ("baseline_top_50.jsonl", "y_score"),
    "Complex model": ("complex_model_top_50.jsonl", "complex_score"),
    "Metadata embedding model": ("embedding_top_50.jsonl", "embedding_score"),
}

for model_name, (filename, score_column) in model_outputs.items():
    output_path = notebooks_dir / "data" / filename
    if not output_path.exists():
        print(f"{model_name}: output not found at {output_path}. Run the pipeline cell first.")
        continue
    results_df = pd.read_json(output_path, lines=True)
    columns = [
        column for column in [
            "rank", "user_name", "game_name", "title", "viewer_count",
            "y_score", "complex_score", "metadata_similarity", "embedding_score",
        ] if column in results_df.columns
    ]
    print(f"\n{model_name} ({len(results_df)} recommendations)")
    display(results_df[columns])


Baseline model (50 recommendations)


,rank,user_name,game_name,title,viewer_count,y_score
0,1,Kamet0,League of Legends,KARMINE CORP 0 vs 0 MOVISTAR KOI | WORLDS 2026...,109134,0.625801
1,2,Caedrel,League of Legends,🔴WORLDS 2026 KOI VS KC ELIMINATION🔴,137490,0.593249
2,3,ibai,League of Legends,#WORLDS2026 | MKOI vs KC | EL PARTIDO DEL AÑO ...,66175,0.575976
3,4,Trymacs,Minecraft,100.000 TNT Exploion Craft Attack Rekord aufst...,31062,0.521966
4,5,otplol_,League of Legends,WORLDS - KC vs MKOI - POUR UNE PLACE EN QUART ...,36421,0.515427
5,6,EDISON,Just Chatting,🎃КОНКУРС КОСТЮМОВ НА ХЕЛЛОУИН! [сейчас],27898,0.499949
6,7,afoninje,Dota 2,World Squad [0:0] CIS Squad | Bo1 | PARI UNIVE...,19762,0.497839
7,8,Riot Games,League of Legends,KC vs MKOI | WORLDS 2026 | Swiss Stage Day 8,28350,0.497634
8,9,Warframe,Warframe,"PMCF Quest Finale | Saturday, October 31 @ 12P...",26387,0.495889
9,10,WarThunder_Esports,War Thunder,Invictus 1-2 VOYAGER | 4v4 Ground | WTCS Pro L...,19416,0.492521



Complex model (50 recommendations)


,rank,user_name,game_name,title,viewer_count,complex_score
0,1,ibai,League of Legends,#WORLDS2026 | MKOI vs KC | EL PARTIDO DEL AÑO ...,66175,0.571629
1,2,otplol_,League of Legends,WORLDS - KC vs MKOI - POUR UNE PLACE EN QUART ...,36421,0.571629
2,3,Kamet0,League of Legends,KARMINE CORP 0 vs 0 MOVISTAR KOI | WORLDS 2026...,109134,0.570420
3,4,Riot Games,League of Legends,KC vs MKOI | WORLDS 2026 | Swiss Stage Day 8,28350,0.550001
4,5,Caedrel,League of Legends,🔴WORLDS 2026 KOI VS KC ELIMINATION🔴,137490,0.543878
5,6,EDISON,Just Chatting,🎃КОНКУРС КОСТЮМОВ НА ХЕЛЛОУИН! [сейчас],27898,0.535959
6,7,Trymacs,Minecraft,100.000 TNT Exploion Craft Attack Rekord aufst...,31062,0.534673
7,8,Warframe,Warframe,"PMCF Quest Finale | Saturday, October 31 @ 12P...",26387,0.530429
8,9,ohnePixel,Just Chatting,🔴JAPAN STORY TIME🔴,22956,0.508772
9,10,deepins02,Just Chatting,[ти таник] | !winline !aliexpress !tg,18323,0.507451



Metadata embedding model (50 recommendations)


,rank,user_name,game_name,title,viewer_count,metadata_similarity,embedding_score
0,1,Caedrel,League of Legends,🔴WORLDS 2026 KOI VS KC ELIMINATION🔴,137490,0,0.30000
1,2,Kamet0,League of Legends,KARMINE CORP 0 vs 0 MOVISTAR KOI | WORLDS 2026...,109134,0,0.29994
2,3,ibai,League of Legends,#WORLDS2026 | MKOI vs KC | EL PARTIDO DEL AÑO ...,66175,0,0.29988
3,4,otplol_,League of Legends,WORLDS - KC vs MKOI - POUR UNE PLACE EN QUART ...,36421,0,0.29982
4,5,Trymacs,Minecraft,100.000 TNT Exploion Craft Attack Rekord aufst...,31062,0,0.29976
5,6,Riot Games,League of Legends,KC vs MKOI | WORLDS 2026 | Swiss Stage Day 8,28350,0,0.29970
6,7,EDISON,Just Chatting,🎃КОНКУРС КОСТЮМОВ НА ХЕЛЛОУИН! [сейчас],27898,0,0.29964
7,8,Warframe,Warframe,"PMCF Quest Finale | Saturday, October 31 @ 12P...",26387,0,0.29958
8,9,ohnePixel,Just Chatting,🔴JAPAN STORY TIME🔴,22956,0,0.29952
9,10,afoninje,Dota 2,World Squad [0:0] CIS Squad | Bo1 | PARI UNIVE...,19762,0,0.29946


## Main outputs

The job writes refreshed JSONL snapshots under `notebooks/data/`, including `baseline_top_50.jsonl`, `complex_model_top_50.jsonl`, and `embedding_top_50.jsonl`. Notebook 00 also exports the collected records to Postgres when `DATABASE_URL` is configured.